In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

True
NVIDIA GeForce GTX 1050 Ti with Max-Q Design
(6, 1)


In [1]:
from ollama import Client
import time

In [3]:
import time, torch

t0 = time.time()
torch.cuda.init()
torch.cuda.synchronize()
t1 = time.time()

print("GPU startup time:", round(t1 - t0, 3), "seconds")


GPU startup time: 0.92 seconds


In [4]:
import torch, time

device = torch.device("cuda")

# 2048 x 2048 matrix multiply
a = torch.randn(2048, 2048, device=device)
b = torch.randn(2048, 2048, device=device)

torch.cuda.synchronize()
t0 = time.time()

for _ in range(20):
    c = a @ b

torch.cuda.synchronize()
t1 = time.time()

print("Time per matmul:", round((t1 - t0)/20, 4), "sec")

Time per matmul: 0.0172 sec


In [6]:

import torch, torch.nn as nn, time

device = torch.device("cuda")

model = nn.Sequential(
    nn.Linear(2048, 2048),
    nn.ReLU(),
    nn.Linear(2048, 2048)
).to(device)

opt = torch.optim.Adam(model.parameters())

x = torch.randn(32, 2048, device=device)

# warm-up
for _ in range(5):
    loss = model(x).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
torch.cuda.synchronize()

# timed loop
t0 = time.time()
for _ in range(20):
    loss = model(x).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
torch.cuda.synchronize()
t1 = time.time()

print("Time per training step:", round((t1 - t0)/20, 4), "sec")


Time per training step: 0.0076 sec


In [12]:
import torch
import torch_geometric
import torch_scatter, torch_sparse, torch_cluster, torch_spline_conv

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())

torch: 2.7.1+cu118
cuda: 11.8
GPU available: True


In [13]:
import torch
import torch_scatter

print("CUDA available:", torch.cuda.is_available())

# try a CUDA scatter operation
try:
    x = torch.randn(5, device='cuda')
    index = torch.tensor([0, 1, 0, 1, 0], device='cuda')
    out = torch_scatter.scatter_add(x, index, dim=0, dim_size=2)
    print("scatter_add works on CUDA:", out)
except Exception as e:
    print("scatter_add failed:", e)


CUDA available: True
scatter_add works on CUDA: tensor([-3.3391, -2.3138], device='cuda:0')
